# Capstone data loader — Fall 2026

All 14 prepared datasets are included in the extracted course packet. Install `requirements.txt` once, then run all cells offline. Keep this notebook, `capstone_data.py`, `data_manifest.json`, and `snapshots/` together. These examples inspect data; they do not fit models or fill missing values. See the handout for the evaluation contract and README for source/cleaning details.

In [1]:
from pathlib import Path
import hashlib
from capstone_data import ROOT, load_dataset, chronological_split, validate_all, load_pems_graph
print('MANIFEST_SHA256', hashlib.sha256((ROOT / 'data_manifest.json').read_bytes()).hexdigest())
print('LOADER_SHA256', hashlib.sha256((ROOT / 'capstone_data.py').read_bytes()).hexdigest())

MANIFEST_SHA256 d21a96a9759c76c6fdd4385ccc43bf48dcdab89ec5913c1dc5107e917db90243
LOADER_SHA256 15fb73e74511a6412685eb91d6441108c9596d0578039760e9f1327471f4a145


## 1. FRED-MD macroeconomic panel

[Original source](https://www.stlouisfed.org/research/economists/mccracken/fred-databases). ID: `fred_md`. Default target: `UNRATE`. Horizon: 3 observations. Units: UNRATE: percent; other variables: source-specific.

**Preparation:** January 2026 vintage, observations 1960–2024; transformation-code row removed and preserved in metadata; levels and missing values retained. Revised-data retrospective benchmark, not a real-time-vintage backtest.

The printed split boundaries use the common 60/20/20 rule. Do not construct target windows across those boundaries.

In [2]:
df = load_dataset('fred_md')
train, validation, test = chronological_split(df)
print('Shape:', df.shape, '| Coverage:', df.index[0], 'to', df.index[-1])
print('Missing cells:', int(df.isna().sum().sum()))
print('Training ends:', train.index[-1], '| Validation ends:', validation.index[-1])
print(df.head(3).to_string(max_cols=6))

Shape: (780, 126) | Coverage: 1960-01-01 00:00:00 to 2024-12-01 00:00:00
Missing cells: 814
Training ends: 1998-12-01 00:00:00 | Validation ends: 2011-12-01 00:00:00
                 RPI  W875RX1  DPCERA3M086SBEA  ...  DTCTHFNM   INVEST  VIXCLSx
time                                            ...                            
1960-01-01  2695.694   2537.6           15.789  ...   14063.0  75.4515      NaN
1960-02-01  2698.789   2539.9           15.858  ...   14144.0  73.3363      NaN
1960-03-01  2703.859   2542.2           16.083  ...   14239.0  71.5296      NaN


## 2. Bitcoin daily volatility

[Original source](https://finance.yahoo.com/quote/BTC-USD/history/). ID: `bitcoin`. Default target: `squared_log_return`. Horizon: 7 observations. Units: squared log return (daily variance proxy; not intraday realized variance).

**Preparation:** Daily adjusted OHLC; explicit auto_adjust=True; natural-log close-to-close return and its square. First return is missing. No rolling target or future-return features precomputed. Bitcoin horizons count calendar days.

The printed split boundaries use the common 60/20/20 rule. Do not construct target windows across those boundaries.

In [3]:
df = load_dataset('bitcoin')
train, validation, test = chronological_split(df)
print('Shape:', df.shape, '| Coverage:', df.index[0], 'to', df.index[-1])
print('Missing cells:', int(df.isna().sum().sum()))
print('Training ends:', train.index[-1], '| Validation ends:', validation.index[-1])
print(df.head(3).to_string(max_cols=6))

Shape: (1827, 7) | Coverage: 2020-01-01 00:00:00 to 2024-12-31 00:00:00
Missing cells: 2
Training ends: 2022-12-31 00:00:00 | Validation ends: 2023-12-31 00:00:00
                  Close         High          Low  ...       Volume  log_return  squared_log_return
time                                               ...                                             
2020-01-01  7200.174316  7254.330566  7174.944336  ...  18565664997         NaN                 NaN
2020-01-02  6985.470215  7212.155273  6935.270020  ...  20802083465   -0.030273            0.000916
2020-01-03  7344.884277  7413.715332  6914.996094  ...  28111481032    0.050172            0.002517


## 3. Jena hourly climate

[Original source](https://www.bgc-jena.mpg.de/wetter/). ID: `jena`. Default target: `T (degC)`. Horizon: 24 observations. Units: temperature: degrees Celsius.

**Preparation:** Average repeated measurements at 327 duplicate timestamps; negative wind measurements become missing. Right-labeled hourly means require six observed ten-minute values per variable. Incomplete hours remain missing. No imputation.

The printed split boundaries use the common 60/20/20 rule. Do not construct target windows across those boundaries.

In [4]:
df = load_dataset('jena')
train, validation, test = chronological_split(df)
print('Shape:', df.shape, '| Coverage:', df.index[0], 'to', df.index[-1])
print('Missing cells:', int(df.isna().sum().sum()))
print('Training ends:', train.index[-1], '| Validation ends:', validation.index[-1])
print(df.head(3).to_string(max_cols=6))

Shape: (70128, 14) | Coverage: 2009-01-01 01:00:00 to 2017-01-01 00:00:00
Missing cells: 1324
Training ends: 2013-10-20 04:00:00 | Validation ends: 2015-05-27 14:00:00
                       p (mbar)  T (degC)    Tpot (K)  ...  wv (m/s)  max. wv (m/s)    wd (deg)
time                                                   ...                                     
2009-01-01 01:00:00  996.523333 -8.261667  265.161667  ...  0.468333       0.940000  177.500000
2009-01-01 02:00:00  996.545000 -8.203333  265.221667  ...  0.323333       0.711667  172.016667
2009-01-01 03:00:00  996.781667 -8.751667  264.653333  ...  0.236667       0.606667  192.966667


## 4. UCI hourly electricity demand

[Original source](https://archive.ics.uci.edu/dataset/321/electricityloaddiagrams20112014). ID: `electricity`. Default target: `MT_001`. Horizon: 24 observations. Units: kWh per hour.

**Preparation:** First 20 client IDs by file order, fixed without target-based selection. 2012–2014. Subtract 15 minutes from interval-end labels; sum four kW readings/4 to obtain kWh per hour, labeled at interval start. An hour becomes known only after it ends. Preserve source Portuguese clock and DST conventions, including structural zeros; do not interpret as UTC.

The printed split boundaries use the common 60/20/20 rule. Do not construct target windows across those boundaries.

In [5]:
df = load_dataset('electricity')
train, validation, test = chronological_split(df)
print('Shape:', df.shape, '| Coverage:', df.index[0], 'to', df.index[-1])
print('Missing cells:', int(df.isna().sum().sum()))
print('Training ends:', train.index[-1], '| Validation ends:', validation.index[-1])
print(df.head(3).to_string(max_cols=6))

Shape: (26304, 20) | Coverage: 2012-01-01 00:00:00 to 2014-12-31 23:00:00
Missing cells: 0
Training ends: 2013-10-19 13:00:00 | Validation ends: 2014-05-26 18:00:00
                       MT_001     MT_002     MT_003  ...      MT_018     MT_019     MT_020
time                                                 ...                                  
2012-01-01 00:00:00  4.124365  22.759602  77.324066  ...  328.274760  15.577889  55.066079
2012-01-01 01:00:00  4.758883  23.115220  77.324066  ...  305.910543  15.452261  51.101322
2012-01-01 02:00:00  4.124365  22.937411  77.324066  ...  276.357827  14.070352  48.458150


## 5. Solar generation at two plants

[Original source](https://www.kaggle.com/datasets/anikannal/solar-power-generation-data). ID: `solar`. Default target: `plant1_ac_power`. Horizon: 4 observations. Units: summed AC_POWER in source units; temperature in degrees Celsius; verify physical power scale before engineering use.

**Preparation:** Parse plant-specific date formats. Within each plant, join weather and generation by timestamp, never by SOURCE_KEY. Sum AC_POWER only when all 22 inverters are observed; otherwise target is missing. Preserve missing weather/time points and observed inverter counts. No cross-time imputation. Only lagged weather is available in an operational forecast.

The printed split boundaries use the common 60/20/20 rule. Do not construct target windows across those boundaries.

In [6]:
df = load_dataset('solar')
train, validation, test = chronological_split(df)
print('Shape:', df.shape, '| Coverage:', df.index[0], 'to', df.index[-1])
print('Missing cells:', int(df.isna().sum().sum()))
print('Training ends:', train.index[-1], '| Validation ends:', validation.index[-1])
print(df.head(3).to_string(max_cols=6))

Shape: (3264, 10) | Coverage: 2020-05-15 00:00:00 to 2020-06-17 23:45:00
Missing cells: 1488
Training ends: 2020-06-04 09:15:00 | Validation ends: 2020-06-11 04:30:00
                     plant1_ac_power  plant1_inverters_observed  plant1_AMBIENT_TEMPERATURE  ...  plant2_AMBIENT_TEMPERATURE  plant2_MODULE_TEMPERATURE  plant2_IRRADIATION
time                                                                                         ...                                                                           
2020-05-15 00:00:00              NaN                       21.0                   25.184316  ...                   27.004764                  25.060789                 0.0
2020-05-15 00:15:00              NaN                       21.0                   25.084589  ...                   26.880811                  24.421869                 0.0
2020-05-15 00:30:00              NaN                       21.0                   24.935753  ...                   26.682055                  24.

## 6. Beijing multi-site air quality

[Original source](https://archive.ics.uci.edu/dataset/501/beijing+multi+site+air+quality+data). ID: `beijing`. Default target: `Guanyuan_PM2.5`. Horizon: 24 observations. Units: PM2.5: micrograms per cubic meter.

**Preparation:** Read all 12 station CSVs from the nested PRSA archive, construct hourly timestamps, align stations, and retain seven numeric measurements per site. Missing measurements remain missing; other sites' future measurements are not known covariates.

The printed split boundaries use the common 60/20/20 rule. Do not construct target windows across those boundaries.

In [7]:
df = load_dataset('beijing')
train, validation, test = chronological_split(df)
print('Shape:', df.shape, '| Coverage:', df.index[0], 'to', df.index[-1])
print('Missing cells:', int(df.isna().sum().sum()))
print('Training ends:', train.index[-1], '| Validation ends:', validation.index[-1])
print(df.head(3).to_string(max_cols=6))

Shape: (35064, 84) | Coverage: 2013-03-01 00:00:00 to 2017-02-28 23:00:00
Missing cells: 17090
Training ends: 2015-07-25 13:00:00 | Validation ends: 2016-05-12 18:00:00
                     Aotizhongxin_PM2.5  Aotizhongxin_PM10  Aotizhongxin_TEMP  ...  Wanshouxigong_DEWP  Wanshouxigong_RAIN  Wanshouxigong_WSPM
time                                                                           ...                                                            
2013-03-01 00:00:00                 4.0                4.0               -0.7  ...               -19.0                 0.0                 2.0
2013-03-01 01:00:00                 8.0                8.0               -1.1  ...               -19.3                 0.0                 4.4
2013-03-01 02:00:00                 7.0                7.0               -1.1  ...               -19.7                 0.0                 4.7


## 7. Online retail daily revenue hierarchy

[Original source](https://archive.ics.uci.edu/dataset/352/online+retail). ID: `retail`. Default target: `selected_countries_total`. Horizon: 7 observations. Units: GBP gross recorded revenue.

**Preparation:** Fixed five-country panel, December 2010–November 2011 (exclude incomplete final December). Keep positive-quantity, positive-price, non-cancellation transactions; aggregate gross recorded revenue in GBP by day/country, retaining duplicate source transactions. Missing country-days become zero recorded revenue conditional on complete transaction reporting; no claim about latent demand or opening status. Total is exactly the sum of selected countries, not all worldwide sales. No customer identifiers retained.

The printed split boundaries use the common 60/20/20 rule. Do not construct target windows across those boundaries.

In [8]:
df = load_dataset('retail')
train, validation, test = chronological_split(df)
print('Shape:', df.shape, '| Coverage:', df.index[0], 'to', df.index[-1])
print('Missing cells:', int(df.isna().sum().sum()))
print('Training ends:', train.index[-1], '| Validation ends:', validation.index[-1])
print(df.head(3).to_string(max_cols=6))

Shape: (365, 6) | Coverage: 2010-12-01 00:00:00 to 2011-11-30 00:00:00
Missing cells: 0
Training ends: 2011-07-07 00:00:00 | Validation ends: 2011-09-18 00:00:00
            United Kingdom  France  Germany     EIRE  Spain  selected_countries_total
time                                                                                 
2010-12-01        54818.08  855.86   261.48   555.38    0.0                  56490.80
2010-12-02        47570.53    0.00   155.35    22.50    0.0                  47748.38
2010-12-03        41308.69  425.38   555.53  2576.85  620.0                  45486.45


## 8. Capital Bikeshare daily demand

[Original source](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset). ID: `bike`. Default target: `cnt`. Horizon: 7 observations. Units: rentals per day.

**Preparation:** Daily series for 2011–2012. Drop identifier and casual/registered counts because those sum to the target. Preserve source-normalized weather and calendar fields. Future observed weather is unavailable: use lagged weather, forecast weather, or explicitly labeled oracle analysis. Calendar fields can be known ahead.

The printed split boundaries use the common 60/20/20 rule. Do not construct target windows across those boundaries.

In [9]:
df = load_dataset('bike')
train, validation, test = chronological_split(df)
print('Shape:', df.shape, '| Coverage:', df.index[0], 'to', df.index[-1])
print('Missing cells:', int(df.isna().sum().sum()))
print('Training ends:', train.index[-1], '| Validation ends:', validation.index[-1])
print(df.head(3).to_string(max_cols=6))

Shape: (731, 12) | Coverage: 2011-01-01 00:00:00 to 2012-12-31 00:00:00
Missing cells: 0
Training ends: 2012-03-13 00:00:00 | Validation ends: 2012-08-06 00:00:00
            season  yr  mnth  ...       hum  windspeed   cnt
time                          ...                           
2011-01-01       1   0     1  ...  0.805833   0.160446   985
2011-01-02       1   0     1  ...  0.696087   0.248539   801
2011-01-03       1   0     1  ...  0.437273   0.248309  1349


## 9. Wikipedia daily article views

[Original source](https://doc.wikimedia.org/generated-data-platform/aqs/analytics-api/reference/page-views.html). ID: `wikipedia`. Default target: `Python_(programming_language)`. Horizon: 7 observations. Units: page views per UTC day.

**Preparation:** Five article titles fixed in advance; English Wikipedia, all-access, user agent category, daily UTC totals 2016–2024. Preserve absent API dates as missing, not zero. Counts apply to these exact titles; redirects/title moves and unusual traffic can change interpretation. No claim to reproduce the Kaggle competition corpus.

The printed split boundaries use the common 60/20/20 rule. Do not construct target windows across those boundaries.

In [10]:
df = load_dataset('wikipedia')
train, validation, test = chronological_split(df)
print('Shape:', df.shape, '| Coverage:', df.index[0], 'to', df.index[-1])
print('Missing cells:', int(df.isna().sum().sum()))
print('Training ends:', train.index[-1], '| Validation ends:', validation.index[-1])
print(df.head(3).to_string(max_cols=6))

Shape: (3288, 5) | Coverage: 2016-01-01 00:00:00 to 2024-12-31 00:00:00
Missing cells: 0
Training ends: 2021-05-25 00:00:00 | Validation ends: 2023-03-14 00:00:00
            Python_(programming_language)  Statistics  Machine_learning  Time_series  Data_science
time                                                                                              
2016-01-01                           3488        1418              1530          478           710
2016-01-02                           4517        1939              1922          613           867
2016-01-03                           3938        2262              2051          692           978


## 10. PEMS-BAY traffic speed

[Original source](https://github.com/liyaguang/DCRNN#data-preparation). ID: `pems_bay`. Default target: `400001`. Horizon: 12 observations. Units: miles per hour.

**Preparation:** First 20 sensors in source order, January–June 2017; source speed values retained. Distances and locations for the full network are supplied separately: subset and reorder them by the returned sensor IDs. Inspect zeros and the source's processing before interpreting sensor outages. Graph models are optional.

The printed split boundaries use the common 60/20/20 rule. Do not construct target windows across those boundaries.

In [11]:
df = load_dataset('pems_bay')
train, validation, test = chronological_split(df)
print('Shape:', df.shape, '| Coverage:', df.index[0], 'to', df.index[-1])
print('Missing cells:', int(df.isna().sum().sum()))
print('Training ends:', train.index[-1], '| Validation ends:', validation.index[-1])
print(df.head(3).to_string(max_cols=6))

Shape: (52128, 20) | Coverage: 2017-01-01 00:00:00 to 2017-06-30 23:55:00
Missing cells: 240
Training ends: 2017-04-19 14:15:00 | Validation ends: 2017-05-25 19:05:00
                     400001  400017  400030  ...  400104  400109  400122
time                                         ...                        
2017-01-01 00:00:00    71.4    67.8    70.5  ...    68.1    68.8    68.2
2017-01-01 00:05:00    71.6    67.5    70.6  ...    67.9    68.4    68.1
2017-01-01 00:10:00    71.6    67.6    70.2  ...    67.5    68.4    68.6


In [12]:
locations, edges = load_pems_graph()
assert list(locations.index) == list(df.columns)
print('Aligned locations:', locations.shape, '| Subset edges:', edges.shape)

Aligned locations: (20, 2) | Subset edges: (54, 3)


## 11. NYC yellow taxi hourly pickups

[Original source](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page). ID: `nyc_taxi`. Default target: `zone_161`. Horizon: 24 observations. Units: recorded pickups per hour.

**Preparation:** January–June 2023. Retain pickup timestamps inside their source file's nominal month, then count pickups in fixed zones 132, 138, 161, 230, 237. Zero means no retained records in that hour/zone, conditional on complete monthly reporting. Local wall-clock labels retain source DST ambiguity; no UTC reconstruction. No drop-off, fare, or trip-outcome predictors.

The printed split boundaries use the common 60/20/20 rule. Do not construct target windows across those boundaries.

In [13]:
df = load_dataset('nyc_taxi')
train, validation, test = chronological_split(df)
print('Shape:', df.shape, '| Coverage:', df.index[0], 'to', df.index[-1])
print('Missing cells:', int(df.isna().sum().sum()))
print('Training ends:', train.index[-1], '| Validation ends:', validation.index[-1])
print(df.head(3).to_string(max_cols=6))

Shape: (4344, 5) | Coverage: 2023-01-01 00:00:00 to 2023-06-30 23:00:00
Missing cells: 0
Training ends: 2023-04-19 13:00:00 | Validation ends: 2023-05-25 18:00:00
                     zone_132  zone_138  zone_161  zone_230  zone_237
time                                                                 
2023-01-01 00:00:00       258        16       195         0       219
2023-01-01 01:00:00       117        26       169        15       235
2023-01-01 02:00:00        14         0       140        45       144


## 12. MIT-BIH record 100 waveform

[Original source](https://physionet.org/content/mitdb/1.0.0/). ID: `ecg`. Default target: `MLII`. Horizon: 36 observations. Units: mV.

**Preparation:** Record 100, both channels in physical units, native 360 Hz sampling, ordered by sample number. Index is elapsed sample count, not a calendar date. One-record short-horizon waveform benchmark only; no arrhythmia-classification or cross-patient generalization claims. Future annotations are not predictors.

The printed split boundaries use the common 60/20/20 rule. Do not construct target windows across those boundaries.

In [14]:
df = load_dataset('ecg')
train, validation, test = chronological_split(df)
print('Shape:', df.shape, '| Coverage:', df.index[0], 'to', df.index[-1])
print('Missing cells:', int(df.isna().sum().sum()))
print('Training ends:', train.index[-1], '| Validation ends:', validation.index[-1])
print(df.head(3).to_string(max_cols=6))

Shape: (650000, 2) | Coverage: 0 to 649999
Missing cells: 0
Training ends: 389999 | Validation ends: 519999
         MLII     V5
sample              
0      -0.145 -0.065
1      -0.145 -0.065
2      -0.145 -0.065


## 13. Annual sunspot activity

[Original source](https://www.statsmodels.org/stable/datasets/generated/sunspots.html). ID: `sunspots`. Default target: `SUNACTIVITY`. Horizon: 5 observations. Units: annual sunspot number.

**Preparation:** Unmodified statsmodels annual sunspot series, 309 years, 1700–2008. Annual data, not the monthly SILSO product.

The printed split boundaries use the common 60/20/20 rule. Do not construct target windows across those boundaries.

In [15]:
df = load_dataset('sunspots')
train, validation, test = chronological_split(df)
print('Shape:', df.shape, '| Coverage:', df.index[0], 'to', df.index[-1])
print('Missing cells:', int(df.isna().sum().sum()))
print('Training ends:', train.index[-1], '| Validation ends:', validation.index[-1])
print(df.head(3).to_string(max_cols=6))

Shape: (309, 1) | Coverage: 1700-01-01 00:00:00 to 2008-01-01 00:00:00
Missing cells: 0
Training ends: 1884-01-01 00:00:00 | Validation ends: 1946-01-01 00:00:00
            SUNACTIVITY
time                   
1700-01-01          5.0
1701-01-01         11.0
1702-01-01         16.0


## 14. Appliance energy and environmental sensors

[Original source](https://archive.ics.uci.edu/dataset/374/appliances+energy+prediction). ID: `appliances`. Default target: `Appliances`. Horizon: 6 observations. Units: Wh per ten-minute interval.

**Preparation:** Continuous ten-minute timeline, January–May 2016. Keep energy plus nine local temperature/humidity sensor pairs. Omit synthetic rv1/rv2 and interpolated airport weather. Use past sensor values only; contemporaneous regression is not a forecast. Source ten-minute averaging remains part of the data provenance.

The printed split boundaries use the common 60/20/20 rule. Do not construct target windows across those boundaries.

In [16]:
df = load_dataset('appliances')
train, validation, test = chronological_split(df)
print('Shape:', df.shape, '| Coverage:', df.index[0], 'to', df.index[-1])
print('Missing cells:', int(df.isna().sum().sum()))
print('Training ends:', train.index[-1], '| Validation ends:', validation.index[-1])
print(df.head(3).to_string(max_cols=6))

Shape: (19735, 20) | Coverage: 2016-01-11 17:00:00 to 2016-05-27 18:00:00
Missing cells: 0
Training ends: 2016-04-02 22:20:00 | Validation ends: 2016-04-30 08:10:00
                     Appliances  lights     T1  ...       RH_8         T9   RH_9
time                                            ...                             
2016-01-11 17:00:00          60      30  19.89  ...  48.900000  17.033333  45.53
2016-01-11 17:10:00          60      30  19.89  ...  48.863333  17.066667  45.56
2016-01-11 17:20:00          50      30  19.89  ...  48.730000  17.000000  45.50


## Computed validation status

A failure raises an exception. This table is generated from the files and manifest.

In [17]:
report = validate_all()
assert len(report) == 14 and report.status.eq('PASS').all()
print(report.to_string(index=False))
print('ALL 14 DATASETS VERIFIED')

    dataset   rows  columns  missing_cells           train_end      validation_end status
    fred_md    780      126            814 1998-12-01 00:00:00 2011-12-01 00:00:00   PASS
    bitcoin   1827        7              2 2022-12-31 00:00:00 2023-12-31 00:00:00   PASS
       jena  70128       14           1324 2013-10-20 04:00:00 2015-05-27 14:00:00   PASS
electricity  26304       20              0 2013-10-19 13:00:00 2014-05-26 18:00:00   PASS
      solar   3264       10           1488 2020-06-04 09:15:00 2020-06-11 04:30:00   PASS
    beijing  35064       84          17090 2015-07-25 13:00:00 2016-05-12 18:00:00   PASS
   pems_bay  52128       20            240 2017-04-19 14:15:00 2017-05-25 19:05:00   PASS
   nyc_taxi   4344        5              0 2023-04-19 13:00:00 2023-05-25 18:00:00   PASS
        ecg 650000        2              0              389999              519999   PASS
   sunspots    309        1              0 1884-01-01 00:00:00 1946-01-01 00:00:00   PASS
     retai